<a href="https://colab.research.google.com/github/gseetharami352005/CSA6301/blob/main/exp7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
import re
from collections import defaultdict

def analyze_login_log(log_data):
    """
    Parses a list of log entries to count failed login attempts per IP address.

    Args:
        log_data (list): A list of strings, each representing a log entry.

    Returns:
        dict: A dictionary where keys are IP addresses and values are the
              count of failed login attempts from that IP.
    """
    failed_attempts_by_ip = defaultdict(int)

    # Regex for IPv4 addresses (more robust, covering valid ranges for each octet)
    ipv4_pattern = r'\b(?:(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\.){3}(?:25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)\b'

    # Regex for IPv6 addresses (covers full, abbreviated with '::', and leading/trailing '::')
    # This pattern aims to be more specific to avoid matching non-IP strings like timestamps.
    ipv6_pattern = (r'\b(?:[0-9a-fA-F]{1,4}:){7}[0-9a-fA-F]{1,4}\b|'  # Full IPv6
                    r'\b(?:[0-9a-fA-F]{1,4}:){1,7}:\b|'  # Abbreviated with :: at the end
                    r'\b:(?:(?::[0-9a-fA-F]{1,4}){1,7})|'  # Abbreviated with :: at the beginning
                    r'\b[0-9a-fA-F]{1,4}(?::[0-9a-fA-F]{1,4}){0,5}::[0-9a-fA-F]{1,4}(?::[0-9a-fA-F]{1,4}){0,5}\b|'  # Abbreviated with :: in the middle
                    r'\b::\b') # Only ::

    # Combine patterns using OR
    ip_pattern = f'(?:{ipv4_pattern}|{ipv6_pattern})'

    # Keywords to identify a failed login attempt
    failed_keywords = ['failed password', 'authentication failure', 'invalid user', 'denied access']

    print("Analyzing login log...")
    for i, entry in enumerate(log_data):
        entry_lower = entry.lower()
        is_failed_attempt = any(keyword in entry_lower for keyword in failed_keywords)

        if is_failed_attempt:
            # Extract all IPs from the log entry
            ips_found = re.findall(ip_pattern, entry)
            if ips_found:
                # For simplicity, we'll associate the failed attempt with the first valid IP found.
                source_ip = ips_found[0]
                failed_attempts_by_ip[source_ip] += 1
                print(f"  Detected failed attempt from {source_ip} in log entry {i+1}: {entry.strip()}")
            else:
                print(f"  Detected failed attempt in log entry {i+1}, but no IP found: {entry.strip()}")

    print("\nLogin log analysis complete.")
    return dict(failed_attempts_by_ip)

# --- Sample Login Log Data --- (In a real scenario, this would be loaded from a file)
sample_login_log = [
    "Oct 26 10:00:01 server sshd[1234]: Accepted password for user1 from 192.168.1.1 port 54321 ssh2",
    "Oct 26 10:00:05 server sshd[1235]: Failed password for user2 from 192.168.1.2 port 12345 ssh2",
    "Oct 26 10:00:10 server sudo[1236]: user3 : authentication failure ; logname=user3 ; uid=1000 ; euid=0 ; tty=/dev/pts/0 ; ruser=user3 ; rhost=192.168.1.3",
    "Oct 26 10:00:15 server sshd[1237]: Invalid user guest from 192.168.1.4 port 67890",
    "Oct 26 10:00:20 server sshd[1238]: Failed password for invalid user admin from 192.168.1.2 port 22222",
    "Oct 26 10:00:25 server sshd[1239]: Accepted password for user4 from 192.168.1.5 port 33333 ssh2",
    "Oct 26 10:00:30 server su[1240]: (to root) user5 on /dev/pts/1: authentication failure",
    "Oct 26 10:00:35 server sshd[1241]: Failed password for root from 10.0.0.100 port 44444",
    "Oct 26 10:00:40 server webapp: User 'baduser' denied access from 172.16.0.1",
    "Oct 26 10:00:45 server sshd[1242]: Accepted publickey for user6 from 2001:0db8::10 port 55555 ssh2",
    "Oct 26 10:00:50 server sshd[1243]: Failed password for testuser from 2001:0db8::10 port 66666"
]

# --- Execute Analysis and Display Results ---
failed_counts = analyze_login_log(sample_login_log)

print("\n--- Failed Login Attempts per IP Address ---")
if failed_counts:
    for ip_address, count in failed_counts.items():
        print(f"  IP Address: {ip_address}, Failed Attempts: {count}")
else:
    print("No failed login attempts found in the log.")

Analyzing login log...
  Detected failed attempt from 192.168.1.2 in log entry 2: Oct 26 10:00:05 server sshd[1235]: Failed password for user2 from 192.168.1.2 port 12345 ssh2
  Detected failed attempt from 192.168.1.3 in log entry 3: Oct 26 10:00:10 server sudo[1236]: user3 : authentication failure ; logname=user3 ; uid=1000 ; euid=0 ; tty=/dev/pts/0 ; ruser=user3 ; rhost=192.168.1.3
  Detected failed attempt from 192.168.1.4 in log entry 4: Oct 26 10:00:15 server sshd[1237]: Invalid user guest from 192.168.1.4 port 67890
  Detected failed attempt from 192.168.1.2 in log entry 5: Oct 26 10:00:20 server sshd[1238]: Failed password for invalid user admin from 192.168.1.2 port 22222
  Detected failed attempt in log entry 7, but no IP found: Oct 26 10:00:30 server su[1240]: (to root) user5 on /dev/pts/1: authentication failure
  Detected failed attempt from 10.0.0.100 in log entry 8: Oct 26 10:00:35 server sshd[1241]: Failed password for root from 10.0.0.100 port 44444
  Detected failed a